In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [2]:
# 1. Loading data
# Load wildfire data
data = pd.read_csv("../data/raw/NFDB_point.csv", sep = ";")
print("Data loaded successfully")

# Load canada data
province_borders = gpd.read_file("../data/raw/lpr_000b21a_e.zip")
print("Zip loaded successfully")

Data loaded successfully
Zip loaded successfully


In [3]:
# 2. Reproject canada data to epsg:4326
province_borders = province_borders.to_crs(epsg = 4326)

# Check which crs the data has
print(province_borders.crs)

EPSG:4326


In [4]:
# 3. Exporting Canada data with epsg 4326
province_borders.to_file("../data/processed/canada_4326.gpkg", driver = "GPKG")
print("Export canada_4326 successful")

Export canada_4326 successful


In [7]:
# 4.1 Data cleaning part 1
# Dropping columns
data_clean = data.drop(["FID",
                        "the_geom",
                        "NFDBFIREID",
                        "NAT_PARK",
                        "FIRENAME",
                        "MONTH",
                        "DAY",
                        "REP_DATE",
                        "OUT_DATE",
                        "FIRE_TYPE",
                        "RESPONSE",
                        "PROTZONE",
                        "MORE_INFO"],
axis = 1) # axis = 1 bedeutet Spalten sind betroffen (axis = 0 würde Zeilen bedeuten)


# Renaming columns
new_names = {
    "SRC_AGENCY": "province",
    "FIRE_ID": "fire_id",
    "LATITUDE": "latitude",
    "LONGITUDE": "longitude",
    "YEAR": "year",
    "CAUSE": "cause",
    "SIZE_HA": "size_ha",
}

data_clean = data_clean.rename(columns=new_names)
data_clean.columns

Index(['province', 'fire_id', 'latitude', 'longitude', 'year', 'size_ha',
       'cause'],
      dtype='str')

In [8]:
# 4.2 Data cleaning part 2
# Dropping rows which have H-PB (= Prescribed Burn) as cause
data_clean = data_clean[data_clean['cause'] != 'H-PB']
print((data_clean['cause'] == 'H-PB').sum())

# Dropping row with wrong longitude
fire_clean = data_clean.drop(data_clean[data_clean["longitude"] > 0].index)

print(f"Rows in data_clean:{len(data_clean)}")
print(f"Rows in fire_clean:{len(fire_clean)}")

0
Rows in data_clean:15138
Rows in fire_clean:15137


In [9]:
# 4.3 Data cleaning part 3
# Check if a column has NaNs
print(data_clean["province"].hasnans)
print(data_clean["fire_id"].hasnans)
print(data_clean["size_ha"].hasnans)
print(data_clean["year"].hasnans)

False
False
False
False


In [11]:
# 4.4 Data cleaning part 4
# Turning the initials of provinces into full names
province_full = {
    "AB": "Alberta",
    "BC": "British Columbia",
    "MB": "Manitoba",
    "NB": "New Brunswick",
    "NL": "Newfoundland and Labrador",
    "NS": "Nova Scotia",
    "NT": "Northwest Territories",
    "ON": "Ontario",
    "PC": "Parc Canada",
    "QC": "Quebec",
    "SK": "Saskatchewan",
    "YT": "Yukon"
}

fire_clean["province"] = fire_clean["province"].map(province_full).fillna(fire_clean["province"])

print(fire_clean["province"].value_counts())

province
Northwest Territories        2524
Saskatchewan                 2392
Manitoba                     2222
Ontario                      1816
British Columbia             1755
Quebec                       1485
Yukon                        1252
Alberta                       930
Parc Canada                   433
Newfoundland and Labrador     271
New Brunswick                  39
Nova Scotia                    18
Name: count, dtype: int64


In [14]:
# 4.5. Data cleaning part 5
# Assigning "Parc Canada" wildfires to the correct provinces

mask = fire_clean["province"] == "Parc Canada" # Only choose rows with "Parc Canada"

# Making GeoDataFrame
geometry = [
    Point(xy) for xy in zip(fire_clean.loc[mask, "longitude"],
                            fire_clean.loc[mask, "latitude"])
]

gdf_points = gpd.GeoDataFrame(
    fire_clean.loc[mask].copy(),
    geometry=geometry,
    crs="EPSG:4326"
)

# IMPORTANT:
# Check name of province column
print(province_borders.columns)

# Spatial join
joined = gpd.sjoin(
    gdf_points,
    province_borders,
    how="left",
    predicate="within"
)

fire_clean.loc[mask, "province"] = joined["PRENAME"].values

print(fire_clean["province"].value_counts())
print((fire_clean["province"] == "Parc Canada").sum())

Index(['PRUID', 'DGUID', 'PRNAME', 'PRENAME', 'PRFNAME', 'PREABBR', 'PRFABBR',
       'LANDAREA', 'geometry'],
      dtype='str')
province
Northwest Territories        2640
Saskatchewan                 2414
Manitoba                     2238
Ontario                      1817
British Columbia             1780
Quebec                       1486
Yukon                        1256
Alberta                      1176
Newfoundland and Labrador     271
New Brunswick                  39
Nova Scotia                    19
Name: count, dtype: int64
0


In [15]:
# 4.6 Data cleaning part 6
# Turning the initials of cause into full names
cause_name = {
    "H": "Human",
    "N": "Natural",
    "U": "Unknown"
}

fire_clean["cause"] = fire_clean["cause"].map(cause_name).fillna(fire_clean["cause"])

In [16]:
# 5. Exporting the cleand fire data
fire_clean.to_csv("../data/processed/fire_clean.csv", index = False)
print("Export successful")

Export successful


In [17]:
# 6. Only keeping the years 2014 - 2023

# To know which years are available
print(fire_clean["year"].unique())

# Dropping row with years I don't need
fire_14_23 = fire_clean.drop(fire_clean[fire_clean["year"] <2014].index)

# To know if the dropping worked
print(fire_14_23["year"].unique())

[2023 2022 2021 2020 2019 2018 2017 2016 2015 2014 2013 2012 2011 2010
 2009 2008 2007 2006 2005 2004 2003 2002 2001 2000 1999 1998 1997 1996
 1995 1994 1993 1992 1991 1990 1989 1988 1987 1986 1985 1984 1983 1982
 1981 1980]
[2023 2022 2021 2020 2019 2018 2017 2016 2015 2014]


In [18]:
# 7. Exporting fire data for 2014-2023
fire_14_23.to_csv("../data/processed/fire_14_23.csv", index = False)
print("Export successful")

Export successful
